<a href="https://colab.research.google.com/github/aannddrree/disciplinaIA/blob/main/Aula_8_Gerenciamento_de_Memoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 17 — Gerenciamento de Memória na Prática

## Simulação de Paginação e Algoritmos de Substituição de Páginas com Python

Nesta aula vamos transformar os conceitos de **gerenciamento de memória** em código.

Ao final, você deverá ser capaz de:

- compreender a relação entre **páginas** e **frames**;
- identificar um **HIT** e um **PAGE FAULT**;
- simular uma memória com quantidade limitada de frames;
- implementar o algoritmo **FIFO**;
- implementar o algoritmo **LRU**;
- comparar algoritmos de substituição de páginas;
- analisar como a quantidade de frames influencia os page faults.

> **Importante:** este notebook é uma **simulação didática**. Não estamos controlando diretamente a memória RAM do computador. Estamos reproduzindo, de maneira simplificada, decisões realizadas pelo sistema operacional.

---
# 1. Relembrando: por que o sistema operacional gerencia memória?

Vários programas podem estar executando ao mesmo tempo e todos precisam utilizar memória.

O sistema operacional precisa decidir:

1. quais informações permanecem na memória RAM;
2. onde elas serão armazenadas;
3. o que fazer quando não há espaço disponível;
4. quais informações podem ser retiradas para dar lugar a outras.

Uma das técnicas utilizadas para organizar esse processo é a **paginação**.

## 1.1 Páginas e frames

De forma simplificada:

- **Página (page):** bloco da memória virtual de um processo.
- **Frame (quadro/moldura):** espaço da memória física capaz de armazenar uma página.

Imagine uma memória RAM disponível para nosso processo com apenas **3 frames**:

```text
┌──────────────┐
│   Frame 0    │
├──────────────┤
│   Frame 1    │
├──────────────┤
│   Frame 2    │
└──────────────┘
```

Agora imagine que um processo faça acessos às páginas:

```text
1 → 2 → 3 → 1 → 4 → 2 → 5 → 1
```

Como existem somente três frames, em algum momento será necessário decidir **qual página deve sair da memória**.

É exatamente aí que entram os **algoritmos de substituição de páginas**.

---
# 2. HIT e PAGE FAULT

Quando um processo solicita uma página, existem duas situações principais.

### HIT

A página solicitada **já está carregada na memória**.

```text
Memória: [1, 2, 3]

Processo solicita página 2

→ Página encontrada
→ HIT
```

### PAGE FAULT

A página solicitada **não está presente na memória**.

```text
Memória: [1, 2, 3]

Processo solicita página 4

→ Página não encontrada
→ PAGE FAULT
```

Se ainda existir um frame livre, a página pode simplesmente ser carregada.

Se todos os frames estiverem ocupados, será necessário escolher uma página para ser substituída.

```text
Acesso a uma página
        │
        ▼
Página está na RAM?
    ┌───┴───┐
   SIM     NÃO
    │        │
   HIT   PAGE FAULT
             │
             ▼
       Existe frame livre?
          ┌──┴──┐
         SIM   NÃO
          │      │
       Carrega  Substitui
                uma página
```

# 3. Nossa primeira memória simulada

Vamos começar criando uma sequência de páginas e definindo que nossa memória possui **3 frames**.

In [ ]:
paginas = [1, 2, 3, 1, 4, 2, 5, 1]
quantidade_frames = 3

print("Sequência de acessos:", paginas)
print("Quantidade de frames:", quantidade_frames)

Sequência de acessos: [1, 2, 3, 1, 4, 2, 5, 1]
Quantidade de frames: 3


A lista `paginas` representa a ordem em que o processo solicita páginas.

Por exemplo:

```text
[1, 2, 3, 1, 4, 2, 5, 1]
```

significa:

```text
acessa página 1
acessa página 2
acessa página 3
acessa página 1 novamente
acessa página 4
...
```

A variável `quantidade_frames = 3` indica que, nesta simulação, podemos manter no máximo **três páginas simultaneamente na memória**.

---
# 4. Algoritmo FIFO

FIFO significa:

> **First In, First Out — Primeiro que entra, primeiro que sai.**

A ideia é simples: quando a memória estiver cheia e uma nova página precisar entrar, removemos a página que está há mais tempo na memória.

Exemplo:

```text
Memória: [1, 2, 3]

Chega página 4.

A página 1 foi a primeira a entrar.

Resultado:

[2, 3, 4]
```

Vamos implementar.

In [ ]:
paginas = [1, 2, 3, 1, 4, 2, 5, 1]
quantidade_frames = 3

memoria = []
faltas = 0
hits = 0

for pagina in paginas:

    if pagina in memoria:
        hits += 1
        print(f"Página {pagina}: HIT        -> {memoria}")

    else:
        faltas += 1

        if len(memoria) < quantidade_frames:
            memoria.append(pagina)
        else:
            memoria.pop(0)
            memoria.append(pagina)

        print(f"Página {pagina}: PAGE FAULT -> {memoria}")

print("\nResultado")
print("Page Faults:", faltas)
print("Hits:", hits)

Página 1: PAGE FAULT -> [1]
Página 2: PAGE FAULT -> [1, 2]
Página 3: PAGE FAULT -> [1, 2, 3]
Página 1: HIT        -> [1, 2, 3]
Página 4: PAGE FAULT -> [2, 3, 4]
Página 2: HIT        -> [2, 3, 4]
Página 5: PAGE FAULT -> [3, 4, 5]
Página 1: PAGE FAULT -> [4, 5, 1]

Resultado
Page Faults: 6
Hits: 2


## 4.1 Entendendo o código FIFO

A memória começa vazia:

```python
memoria = []
```

Quando verificamos:

```python
if pagina in memoria:
```

estamos perguntando:

> A página solicitada já está carregada?

Se estiver, temos um **HIT**.

Caso contrário, temos um **PAGE FAULT**.

Quando a memória está cheia:

```python
memoria.pop(0)
```

remove o elemento da posição `0`, isto é, a página mais antiga da lista.

Depois:

```python
memoria.append(pagina)
```

coloca a nova página no final.

Assim, a própria lista representa a ordem de chegada das páginas.

---
# 5. Transformando FIFO em uma função

Funções tornam o programa reutilizável. Assim podemos testar diferentes sequências e diferentes quantidades de frames sem repetir todo o código.

In [ ]:
def fifo(paginas, quantidade_frames, mostrar_passos=True):
    memoria = []
    faltas = 0
    hits = 0

    for pagina in paginas:

        if pagina in memoria:
            hits += 1
            resultado = "HIT"

        else:
            faltas += 1
            resultado = "PAGE FAULT"

            if len(memoria) >= quantidade_frames:
                memoria.pop(0)

            memoria.append(pagina)

        if mostrar_passos:
            print(
                f"Página: {pagina:<2} | "
                f"Memória: {str(memoria):<15} | "
                f"{resultado}"
            )

    return faltas, hits

In [ ]:
paginas = [1, 2, 3, 1, 4, 2, 5, 1]

faltas, hits = fifo(paginas, 3)

print("\n--- RESULTADO FIFO ---")
print("Page Faults:", faltas)
print("Hits:", hits)

Página: 1  | Memória: [1]             | PAGE FAULT
Página: 2  | Memória: [1, 2]          | PAGE FAULT
Página: 3  | Memória: [1, 2, 3]       | PAGE FAULT
Página: 1  | Memória: [1, 2, 3]       | HIT
Página: 4  | Memória: [2, 3, 4]       | PAGE FAULT
Página: 2  | Memória: [2, 3, 4]       | HIT
Página: 5  | Memória: [3, 4, 5]       | PAGE FAULT
Página: 1  | Memória: [4, 5, 1]       | PAGE FAULT

--- RESULTADO FIFO ---
Page Faults: 6
Hits: 2


### Experimente

Altere a quantidade de frames abaixo.

Observe o que acontece com o número de **page faults**.

In [ ]:
for frames in [2, 3, 4, 5]:
    faltas, hits = fifo(paginas, frames, mostrar_passos=False)
    print(f"{frames} frames -> Page Faults: {faltas} | Hits: {hits}")

2 frames -> Page Faults: 8 | Hits: 0
3 frames -> Page Faults: 6 | Hits: 2
4 frames -> Page Faults: 6 | Hits: 2
5 frames -> Page Faults: 5 | Hits: 3


---
# 6. Algoritmo LRU

LRU significa:

> **Least Recently Used — Menos recentemente utilizada.**

Diferentemente do FIFO, o LRU não pergunta qual página entrou primeiro.

Ele pergunta:

> **Qual página está há mais tempo sem ser utilizada?**

Considere:

```text
Memória: [1, 2, 3]
```

Se acessarmos novamente a página `1`, ela passa a ser considerada a mais recentemente utilizada.

Podemos representar a ordem assim:

```text
[2, 3, 1]
 ↑       ↑
menos   mais
recente recente
```

Se agora chegar a página `4`, a página `2` será removida:

```text
[3, 1, 4]
```

# 7. Implementando LRU em Python

Nesta implementação, manteremos a página menos recentemente utilizada no início da lista e a mais recentemente utilizada no final.

In [ ]:
def lru(paginas, quantidade_frames, mostrar_passos=True):
    memoria = []
    faltas = 0
    hits = 0

    for pagina in paginas:

        if pagina in memoria:
            hits += 1
            resultado = "HIT"

            # A página acabou de ser utilizada.
            # Portanto, ela se torna a mais recente.
            memoria.remove(pagina)
            memoria.append(pagina)

        else:
            faltas += 1
            resultado = "PAGE FAULT"

            if len(memoria) >= quantidade_frames:
                memoria.pop(0)

            memoria.append(pagina)

        if mostrar_passos:
            print(
                f"Página: {pagina:<2} | "
                f"Memória: {str(memoria):<15} | "
                f"{resultado}"
            )

    return faltas, hits

In [ ]:
paginas = [1, 2, 3, 1, 4, 2, 5, 1]

faltas, hits = lru(paginas, 3)

print("\n--- RESULTADO LRU ---")
print("Page Faults:", faltas)
print("Hits:", hits)

Página: 1  | Memória: [1]             | PAGE FAULT
Página: 2  | Memória: [1, 2]          | PAGE FAULT
Página: 3  | Memória: [1, 2, 3]       | PAGE FAULT
Página: 1  | Memória: [2, 3, 1]       | HIT
Página: 4  | Memória: [3, 1, 4]       | PAGE FAULT
Página: 2  | Memória: [1, 4, 2]       | PAGE FAULT
Página: 5  | Memória: [4, 2, 5]       | PAGE FAULT
Página: 1  | Memória: [2, 5, 1]       | PAGE FAULT

--- RESULTADO LRU ---
Page Faults: 7
Hits: 1


## 7.1 A diferença fundamental

### FIFO

Escolhe a página que **entrou primeiro**.

```text
Pergunta:
"Qual página está há mais tempo dentro da memória?"
```

### LRU

Escolhe a página que **não é utilizada há mais tempo**.

```text
Pergunta:
"Qual página foi usada há mais tempo?"
```

Por isso, um HIT não altera a ordem do FIFO, mas pode alterar a ordem utilizada pelo LRU.

---
# 8. Comparando FIFO e LRU

Agora utilizaremos uma sequência clássica de referências:

```text
7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2
```

Vamos comparar os algoritmos utilizando diferentes quantidades de frames.

In [ ]:
paginas_teste = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]

print(f"{'Algoritmo':<12} {'Frames':<8} {'Faults':<8} {'Hits':<8}")
print("-" * 40)

for frames in [3, 4, 5]:
    faltas_fifo, hits_fifo = fifo(paginas_teste, frames, False)
    faltas_lru, hits_lru = lru(paginas_teste, frames, False)

    print(f"{'FIFO':<12} {frames:<8} {faltas_fifo:<8} {hits_fifo:<8}")
    print(f"{'LRU':<12} {frames:<8} {faltas_lru:<8} {hits_lru:<8}")

Algoritmo    Frames   Faults   Hits    
----------------------------------------
FIFO         3        10       3       
LRU          3        9        4       
FIFO         4        7        6       
LRU          4        6        7       
FIFO         5        6        7       
LRU          5        6        7       


## 8.1 Taxa de page faults

Além da quantidade absoluta, podemos calcular:

\[
\text{Taxa de Page Faults} =
\frac{\text{Quantidade de Page Faults}}
{\text{Quantidade total de acessos}}
\times 100
\]

Quanto menor a taxa, melhor foi o desempenho do algoritmo **para aquela sequência de acessos**.

In [ ]:
paginas_teste = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]

for nome, algoritmo in [("FIFO", fifo), ("LRU", lru)]:
    faltas, hits = algoritmo(paginas_teste, 3, False)
    total = faltas + hits

    taxa_fault = (faltas / total) * 100
    taxa_hit = (hits / total) * 100

    print(nome)
    print(f"Taxa de Page Faults: {taxa_fault:.2f}%")
    print(f"Taxa de Hits:        {taxa_hit:.2f}%")
    print()

FIFO
Taxa de Page Faults: 76.92%
Taxa de Hits:        23.08%

LRU
Taxa de Page Faults: 69.23%
Taxa de Hits:        30.77%



---
# 9. Anomalia de Belady

Seria natural imaginar:

> "Se eu aumentar a quantidade de frames, o número de page faults sempre diminuirá."

Isso **não é necessariamente verdade para todos os algoritmos**.

O FIFO pode apresentar a chamada **Anomalia de Belady**: em determinadas sequências, aumentar a quantidade de frames pode **aumentar** o número de page faults.

Uma sequência clássica para observar o fenômeno é:

```text
1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5
```

Execute a próxima célula.

In [ ]:
belady = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]

for frames in [3, 4]:
    faltas, hits = fifo(belady, frames, False)
    print(f"FIFO com {frames} frames -> {faltas} Page Faults")

FIFO com 3 frames -> 9 Page Faults
FIFO com 4 frames -> 10 Page Faults


### Pense antes de continuar

Compare os resultados com 3 e 4 frames.

**Pergunta:** adicionar um frame melhorou o desempenho do FIFO nessa sequência?

Esse resultado mostra por que não devemos avaliar um algoritmo apenas pela intuição.

---
# 10. Desafio extra — Algoritmo Ótimo (OPT)

O algoritmo **Ótimo** substitui a página cujo próximo acesso acontecerá **mais distante no futuro**.

Exemplo:

```text
Memória atual: [1, 2, 3]

Próximos acessos:
2, 4, 1, 5, 3
```

Entre `1`, `2` e `3`:

- página `2` será usada imediatamente;
- página `1` será usada depois;
- página `3` será usada mais tarde.

Logo, seria interessante retirar a página `3`.

### Problema

Para fazer essa escolha perfeitamente, o sistema precisaria conhecer **o futuro**.

Por isso, o algoritmo Ótimo é muito útil como **referência teórica e para comparação**, mas não pode ser implementado de forma perfeita por um sistema operacional real para uma sequência futura desconhecida.

In [ ]:
def otimo(paginas, quantidade_frames, mostrar_passos=True):
    memoria = []
    faltas = 0
    hits = 0

    for i, pagina in enumerate(paginas):

        if pagina in memoria:
            hits += 1
            resultado = "HIT"

        else:
            faltas += 1
            resultado = "PAGE FAULT"

            if len(memoria) < quantidade_frames:
                memoria.append(pagina)

            else:
                futuro = paginas[i + 1:]
                distancias = {}

                for pagina_memoria in memoria:
                    if pagina_memoria in futuro:
                        distancias[pagina_memoria] = futuro.index(pagina_memoria)
                    else:
                        # Se nunca mais será usada, é a melhor candidata a sair.
                        distancias[pagina_memoria] = float("inf")

                pagina_remover = max(distancias, key=distancias.get)
                memoria.remove(pagina_remover)
                memoria.append(pagina)

        if mostrar_passos:
            print(
                f"Página: {pagina:<2} | "
                f"Memória: {str(memoria):<15} | "
                f"{resultado}"
            )

    return faltas, hits

In [ ]:
paginas_teste = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]

for nome, algoritmo in [
    ("FIFO", fifo),
    ("LRU", lru),
    ("Ótimo", otimo)
]:
    faltas, hits = algoritmo(paginas_teste, 3, False)
    print(f"{nome:<8} -> Page Faults: {faltas} | Hits: {hits}")

FIFO     -> Page Faults: 10 | Hits: 3
LRU      -> Page Faults: 9 | Hits: 4
Ótimo    -> Page Faults: 7 | Hits: 6


---
# 11. Atividade prática

## Simulador de Gerenciamento de Memória

Em dupla, desenvolva ou adapte um programa Python capaz de simular algoritmos de substituição de páginas.

O programa deverá:

1. receber uma **sequência de páginas**;
2. receber a **quantidade de frames**;
3. executar o algoritmo **FIFO**;
4. executar o algoritmo **LRU**;
5. mostrar cada acesso realizado;
6. indicar se ocorreu `HIT` ou `PAGE FAULT`;
7. mostrar a situação da memória após cada acesso;
8. apresentar ao final:
   - quantidade de page faults;
   - quantidade de hits;
   - taxa de page faults;
   - taxa de hits;
9. comparar os resultados dos dois algoritmos.

### Exemplo de entrada

```text
Quantidade de frames: 3
Páginas: 1 2 3 4 1 2 5 1 2 3 4 5
```

### Exemplo de saída esperada

```text
Página 1 -> [1]       PAGE FAULT
Página 2 -> [1, 2]    PAGE FAULT
Página 3 -> [1, 2, 3] PAGE FAULT
...
```

Ao final:

```text
FIFO
Page Faults: ...
Hits: ...

LRU
Page Faults: ...
Hits: ...
```

# 12. Código inicial para a atividade

Use a célula abaixo como ponto de partida.

> Tente completar/adaptar o programa antes de consultar as implementações anteriores.

In [ ]:
entrada = input("Digite as páginas separadas por espaço: ")
paginas_usuario = [int(valor) for valor in entrada.split()]

quantidade_frames = int(input("Digite a quantidade de frames: "))

print("\nSequência:", paginas_usuario)
print("Frames:", quantidade_frames)

# TODO:
# 1. Execute FIFO
# 2. Execute LRU
# 3. Mostre Page Faults e Hits
# 4. Calcule as taxas
# 5. Compare os resultados

Digite as páginas separadas por espaço: 1 2 3
Digite a quantidade de frames: 3

Sequência: [1, 2, 3]
Frames: 3


---
# 13. Questões para análise

Depois de executar os experimentos, responda:

**1.** Qual é a diferença entre um HIT e um PAGE FAULT?

**2.** Qual é a principal diferença entre FIFO e LRU?

**3.** Um page fault significa necessariamente que existe um erro no programa? Explique.

**4.** Aumentar a quantidade de frames sempre reduz o número de page faults no FIFO? Utilize o experimento da Anomalia de Belady para justificar.

**5.** Em quais situações FIFO e LRU podem produzir resultados diferentes?

**6.** Por que o algoritmo Ótimo normalmente apresenta um bom resultado?

**7.** Por que um sistema operacional real não consegue utilizar perfeitamente o algoritmo Ótimo?

**8.** Execute pelo menos três sequências diferentes e registre qual algoritmo apresentou menos page faults em cada uma.

---
# 14. Desafio avançado ⭐

Escolha **um** dos desafios:

### Desafio A — Implementar OPT sozinho

Reescreva o algoritmo Ótimo sem copiar a implementação fornecida neste notebook.

### Desafio B — Criar uma interface

Crie um pequeno menu:

```text
===== SIMULADOR DE MEMÓRIA =====

1 - FIFO
2 - LRU
3 - Ótimo
4 - Comparar todos
0 - Sair
```

### Desafio C — Gerar uma sequência automaticamente

Utilize Python para criar 30 acessos aleatórios entre páginas `0` e `9` e compare os algoritmos.

### Desafio D — Visualização

Apresente uma tabela mostrando, para cada acesso:

```text
Página | Frame 1 | Frame 2 | Frame 3 | Resultado
```

Isso aproxima a saída do programa das tabelas utilizadas nos exercícios teóricos de substituição de páginas.

In [ ]:
# Exemplo para o Desafio C

import random

random.seed(42)

paginas_aleatorias = [random.randint(0, 9) for _ in range(30)]

print("Sequência gerada:")
print(paginas_aleatorias)

print("\nComparação com 3 frames:")

for nome, algoritmo in [
    ("FIFO", fifo),
    ("LRU", lru),
    ("Ótimo", otimo)
]:
    faltas, hits = algoritmo(paginas_aleatorias, 3, False)
    print(f"{nome:<8} -> Faults: {faltas:<3} | Hits: {hits}")

Sequência gerada:
[1, 0, 4, 3, 3, 2, 1, 8, 1, 9, 6, 0, 0, 1, 3, 3, 8, 9, 0, 8, 3, 8, 6, 3, 7, 9, 4, 0, 2, 6]

Comparação com 3 frames:
FIFO     -> Faults: 24  | Hits: 6
LRU      -> Faults: 23  | Hits: 7
Ótimo    -> Faults: 17  | Hits: 13


---
# 15. Conclusão

Nesta aula vimos na prática o fluxo:

```text
Programa solicita uma página
           ↓
A página está na memória?
      ┌────┴────┐
     SIM       NÃO
      ↓          ↓
     HIT     PAGE FAULT
                 ↓
         Existe frame livre?
            ┌────┴────┐
           SIM       NÃO
            ↓          ↓
         Carrega    Escolhe página
                    para substituir
                         ↓
                  FIFO / LRU / OPT
```

## O que você deve levar desta aula

- A memória física possui capacidade limitada.
- A paginação divide a memória em unidades de tamanho fixo.
- Um **HIT** ocorre quando a página solicitada já está disponível na memória.
- Um **PAGE FAULT** ocorre quando a página solicitada não está presente.
- Se a memória estiver cheia, um algoritmo decide qual página será substituída.
- **FIFO** considera a ordem de entrada.
- **LRU** considera o histórico recente de utilização.
- **OPT** considera acessos futuros e serve principalmente como referência teórica.
- O desempenho depende tanto do algoritmo quanto do padrão de acesso às páginas.

> **Pergunta final:** se você estivesse projetando um sistema operacional, que informações tentaria utilizar para prever quais páginas provavelmente serão necessárias novamente?